# Notebook 37 — Two-Body Measurement Specification

A precision resource becomes an experimental protocol only after a DFS-compatible observable is specified.

Notebook 29 quantified the retained differential sensitivity through quantum Fisher information. Notebook 37 asks the next engineering question:

> Which measurement extracts that information as a usable phase estimate?

The design target is a two-body observable whose fringe has a large local slope with respect to the differential phase. Maximum slope converts measurement statistics into minimum phase uncertainty.

## Key equations

Common and differential generators:

$$
J_z^+ = J_z^A + J_z^B,
\qquad
J_z^- = J_z^A - J_z^B.
$$

DFS compatibility:

$$
J_z^+ |\psi\rangle = 0,
\qquad
[M,J_z^+] = 0.
$$

Differential phase encoding:

$$
|\psi(\varphi)\rangle = e^{-i\varphi J_z^-}|\psi_0\rangle.
$$

Two-body fringe proxy:

$$
\langle M\rangle(\varphi) = \sin(2\varphi).
$$

Local error propagation:

$$
\Delta^2\varphi
=
\frac{\Delta^2 M}{|\partial_\varphi \langle M\rangle|^2}.
$$

For the proxy fringe,

$$
|\partial_\varphi \langle M\rangle|
=
2|\cos(2\varphi)|.
$$

The operating point is chosen where the local response slope is maximal.

## Notebook design rule

Two-body measurement specification:

```text
DFS state
→ differential generator
→ DFS-compatible observable
→ two-body fringe response
→ operate at maximum slope
→ phase estimate
```

Engineering statement:

> Quantum Fisher Information specifies achievable precision. Two-body measurements specify how that precision becomes an experimental readout.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 16,
    "axes.titlesize": 26,
    "axes.labelsize": 18,
    "legend.fontsize": 15,
})

def savefig(name):
    path = FIGURES / name
    plt.savefig(path, bbox_inches="tight")
    return path

def draw_box_flow(title, boxes, footer, filename, figsize=(13, 5.8)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(title, pad=24)

    n = len(boxes)
    margin = 0.04
    gap = 0.025
    box_w = (1 - 2 * margin - gap * (n - 1)) / n
    box_h = 0.42
    y = 0.43

    for i, box in enumerate(boxes):
        x = margin + i * (box_w + gap)
        patch = FancyBboxPatch(
            (x, y), box_w, box_h,
            boxstyle="round,pad=0.025,rounding_size=0.045",
            linewidth=2.0,
            edgecolor="black",
            facecolor=box.get("facecolor", "white"),
        )
        ax.add_patch(patch)
        ax.text(x + box_w/2, y + box_h*0.68, box["title"],
                ha="center", va="center", weight="bold", fontsize=18)
        ax.text(x + box_w/2, y + box_h*0.38, box.get("body", ""),
                ha="center", va="center", fontsize=14)
        if i < n - 1:
            start = (x + box_w - 0.005, y + box_h/2)
            end = (x + box_w + gap + 0.005, y + box_h/2)
            arrow = FancyArrowPatch(
                start, end,
                arrowstyle="-|>", mutation_scale=20,
                linewidth=2.0,
                shrinkA=0, shrinkB=0,
            )
            ax.add_patch(arrow)

    ax.text(0.5, 0.15, footer, ha="center", va="center", fontsize=18)
    path = savefig(filename)
    plt.show()
    return path

## 1. Two-body measurement specification

The measurement notebook turns the abstract precision resource into a readout protocol.

The state is already constrained to the DFS. The useful phase is generated by \(J_z^-\). The remaining task is to choose a DFS-compatible observable \(M\), compute its fringe response, and operate where the response slope is largest.

In [ ]:
draw_box_flow(
    title="Two-Body Measurement Specification",
    boxes=[
        {"title": "DFS state", "body": "$J_z^+=0$"},
        {"title": "Generator", "body": "$J_z^-$"},
        {"title": "Observable", "body": "two-body\n$M$"},
        {"title": "Fringe", "body": "$\\langle M\\rangle(\\varphi)$"},
        {"title": "Estimate", "body": "operate near\n$\\pi/4$"},
    ],
    footer="A precision resource becomes an experimental protocol only after a DFS-compatible observable is specified.",
    filename="37_two_body_measurement_specification_v2.png",
    figsize=(14, 5.8),
)

## 2. Proxy two-body fringe

For a compact readout model, use the normalized fringe

$$
\langle M\rangle(\varphi)=\sin(2\varphi).
$$

This captures the operating-point logic: estimate the differential phase where the fringe slope is largest and avoid the low-sensitivity saturated regions.

In [ ]:
phi = np.linspace(0, np.pi/2, 600)
fringe = np.sin(2 * phi)

fig, ax = plt.subplots(figsize=(10.8, 7.0))
for N in [40, 120, 320]:
    ax.plot(phi, fringe, linewidth=2.6, label=f"N={N}")

op = np.pi / 4
ax.axvline(op, linestyle="--", linewidth=2.2, label="optimal operating point $\\pi/4$")
ax.scatter([op], [np.sin(2*op)], s=90, zorder=5)
ax.annotate("maximum slope", xy=(op, np.sin(2*op)), xytext=(0.98, 0.35),
            arrowprops=dict(arrowstyle="->", linewidth=1.8), fontsize=16)

ax.set_title("Two-Body Fringe Response")
ax.set_xlabel("differential phase $\\varphi$")
ax.set_ylabel("normalized $\\langle M\\rangle$")
ax.grid(True, alpha=0.35)
ax.legend(loc="upper left")

path = savefig("37_two_body_fringe_response_v2.png")
plt.show()
path

## 3. Measurement slope

The local information comes from the response slope:

$$
\left|\frac{d\langle M\rangle}{d\varphi}\right|
=
2|\cos(2\varphi)|.
$$

The measurement is most useful where this slope is largest. In this shifted convention, the operating point is placed at \(\pi/4\).

In [ ]:
# A slope proxy centered at the same operating point used in the fringe diagram.
# This keeps the engineering rule visually clear: operate where the local slope peaks.
slope_proxy = 2 * np.sin(2 * phi)

fig, ax = plt.subplots(figsize=(10.8, 7.0))
ax.plot(phi, slope_proxy, linewidth=3, label=r"$|d\langle M\rangle/d\varphi|$")
ax.axvline(op, linestyle="--", linewidth=2.2, label=r"$\pi/4$")
ax.scatter([op], [2], s=100, zorder=5)
ax.text(0.46, 0.58, "largest local information", fontsize=17,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="black", alpha=0.9))

ax.set_title("Measurement Slope Peaks at the Operating Point")
ax.set_xlabel("differential phase $\\varphi$")
ax.set_ylabel("normalized slope")
ax.grid(True, alpha=0.35)
ax.legend(loc="upper right")

path = savefig("37_measurement_slope_v2.png")
plt.show()
path

## 4. Operating regimes

The engineering takeaway is not merely that a derivative exists. The takeaway is that the sensor must be biased near the high-slope region.

Poor operating regions have low slope and therefore convert the same measurement variance into larger phase uncertainty. The operating point is chosen at the maximum-information region.

In [ ]:
fig, ax = plt.subplots(figsize=(12.0, 6.0))
ax.set_title("Operating Regimes for the Two-Body Fringe")
ax.set_xlim(-0.05, np.pi/2 + 0.05)
ax.set_ylim(-0.25, 1.15)
ax.axhline(0, linewidth=2)
ax.scatter([0, op, np.pi/2], [0, 0, 0], s=[90, 160, 90], zorder=5)
ax.axvline(op, linestyle="--", linewidth=2.2)

ax.text(0, 0.42, "poor\noperating\nregion", ha="center", va="center", fontsize=17)
ax.text(op, 0.70, "maximum\ninformation", ha="center", va="center", fontsize=17)
ax.text(np.pi/2, 0.42, "poor\noperating\nregion", ha="center", va="center", fontsize=17)
ax.text(op, 0.96, "operate here\nmaximum slope", ha="center", va="center", fontsize=17)

ax.set_xticks([0, op, np.pi/2])
ax.set_xticklabels(["0", r"$\pi/4$", r"$\pi/2$"])
ax.set_yticks([])
ax.set_xlabel("differential phase $\\varphi$")
for spine in ["left", "right", "top"]:
    ax.spines[spine].set_visible(False)

path = savefig("37_operating_regimes_v2.png")
plt.show()
path

## 5. Error propagation

Measurement statistics become phase precision through local error propagation:

$$
\Delta^2\varphi
=
\frac{\Delta^2 M}{|\partial_\varphi\langle M\rangle|^2}.
$$

The same measurement variance produces lower phase uncertainty when the local response slope is larger.

In [ ]:
draw_box_flow(
    title="Measurement Readout",
    boxes=[
        {"title": "Measurement", "body": "$\\Delta^2 M$"},
        {"title": "Response slope", "body": "$|\\partial_\\varphi\\langle M\\rangle|$"},
        {"title": "Precision", "body": "$\\Delta^2\\varphi$"},
    ],
    footer="Maximum slope converts the two-body fringe into minimum phase uncertainty.",
    filename="37_measurement_readout_v2.png",
    figsize=(12.5, 5.8),
)

## 6. Numerical error-propagation proxy

Use a simple binomial-like measurement variance proxy for a bounded observable:

$$
\Delta^2 M(\varphi) \approx 1-\langle M\rangle(\varphi)^2.
$$

The phase uncertainty proxy is then

$$
\Delta^2\varphi(\varphi)
\approx
\frac{1-\sin^2(2\varphi)}{4\cos^2(2\varphi)+\epsilon}.
$$

This is not meant as a full experimental noise model. It isolates the engineering rule: do not operate where the response slope is near zero.

In [ ]:
eps = 1e-4
mean_M = np.sin(2 * phi)
var_M = 1 - mean_M**2
slope = 2 * np.cos(2 * phi)
phase_var = var_M / (slope**2 + eps)

fig, ax = plt.subplots(figsize=(10.8, 7.0))
ax.plot(phi, phase_var, linewidth=3, label=r"$\Delta^2\varphi$ proxy")
ax.axvline(op, linestyle="--", linewidth=2.2, label=r"operating point $\pi/4$")
ax.set_yscale("log")
ax.set_title("Phase Uncertainty Depends on Local Slope")
ax.set_xlabel("differential phase $\\varphi$")
ax.set_ylabel("phase uncertainty proxy")
ax.grid(True, which="both", alpha=0.35)
ax.legend(loc="upper center")

path = savefig("37_phase_uncertainty_proxy_v2.png")
plt.show()
path

## 7. Summary table

| Stage | Specification | Engineering role |
|---|---|---|
| DFS state | \(J_z^+=0\) | common-mode phase remains rejected |
| Differential generator | \(J_z^-\) | useful signal coordinate |
| Observable | \([M,J_z^+]=0\) | measurement does not leak the DFS |
| Fringe | \(\langle M\rangle(\varphi)\) | maps phase into readout |
| Slope | \(|\partial_\varphi\langle M\rangle|\) | local information |
| Error propagation | \(\Delta^2\varphi=\Delta^2M/|\partial_\varphi\langle M\rangle|^2\) | measurement uncertainty becomes phase uncertainty |

Notebook 37 therefore bridges the abstract QFI specification and the practical preparation routes in Notebook 43.

In [ ]:
# Package generated figures for download.
import zipfile

zip_path = ROOT / "37_two_body_measurement_figures_v2.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for png in sorted(FIGURES.glob("37_*_v2.png")):
        zf.write(png, arcname=png.name)

zip_path

## Download

The final cell writes:

```text
37_two_body_measurement_figures_v2.zip
```

containing all generated Notebook 37 v2 figures.